In [ ]:
import pandas as pd
import lyricsgenius
import time
import os
import tarfile
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")
genius = lyricsgenius.Genius(os.getenv("GENIUS_ACCESS_TOKEN"), timeout=15, retries=3)

In [3]:
tags_file = "../data/processed/lastfm_extracted_tags.pkl"
tags_df = pd.read_pickle(tags_file)
existing_df = pd.read_csv("final_df.csv", index_col=0)

In [4]:
tags_df['match_key'] = tags_df['artist'].astype(str).str.lower() + " - " + tags_df['title'].astype(str).str.lower()
existing_df['match_key'] = existing_df['artist'].astype(str).str.lower() + " - " + existing_df['song'].astype(str).str.lower()

unprocessed_df = tags_df[~tags_df['match_key'].isin(existing_df['match_key'])].copy()
unprocessed_df = unprocessed_df.drop(columns=['match_key'])

print(f"Total remaining tracks to process: {len(unprocessed_df)}")

Total remaining tracks to process: 980466


In [6]:
BATCH_SIZE = 500
batch_df = unprocessed_df.head(BATCH_SIZE).copy()
new_records = []

print(f"Starting batch extraction for {BATCH_SIZE} tracks...")

for index, row in batch_df.iterrows():
    try:
        song = genius.search_song(row['title'], row['artist'])
        
        if song and song.lyrics:
            new_records.append({
                'artist': row['artist'],
                'song': row['title'],
                'link': song.url,
                'text': song.lyrics,
                'clean_artist': str(row['artist']).lower(),
                'clean_title': str(row['title']).lower(),
                'tags': row['tags'],
                'genres': []
            })
            
        time.sleep(0.5) 
        
    except Exception as e:
        print(f"API Error on {row['artist']} - {row['title']}: {e}")
        time.sleep(5)

if new_records:
    append_df = pd.DataFrame(new_records)
    
    append_df.to_csv("final_df.csv", mode='a', header=False, index=True)
    print(f"Successfully processed and appended {len(append_df)} new tracks to final_df.csv.")
else:
    print("No valid lyrics found in this batch.")

Starting batch extraction for 500 tracks...
Successfully processed and appended 274 new tracks to final_df.csv.
